# NLU: joint intent + TRAIT slot model (Phase 6)

Fine-tunes DistilBERT with an intent head on `[CLS]` and a BIO tag head on the tokens, on the 2,624 training messages
from `data/nlu/corpus/` (7 intents: `request_diss`, `another_one`, `make_meaner`, `make_softer`, `greet`, `goodbye`,
`out_of_scope`), as a small sweep over learning rates and seeds (chosen on val only). Then each TRAIT span the model
finds is mapped to a trait in `traits.json` (`nlu/normalize.py`): a trait name or synonym first, otherwise the nearest
labelled training spans and trait descriptions by sentence embedding (all-MiniLM-L6-v2), otherwise unknown.

The test set is reported with training-trait and held-out-trait messages apart: intent accuracy, slot F1 (by exact span
match, per word and by overlap, since the LLM chose span boundaries loosely), trait accuracy of the normalization on the
gold spans and end to end. The sweep and the evaluation are in `nlu/train_nlu.py`, the model and training loop in
`nlu/joint_model.py`.

**On Colab:**
1. On your machine, from the project root: `python nlu/make_colab_zip.py`. It writes `nlu_colab.zip` (about 0.3 MB:
   the `nlu/` code, `utils.py`, `config.yaml`, `traits.json`, the NLU corpus and the two files with the trait groups).
2. Runtime → Change runtime type → **T4 GPU**, then Run all. The first cell asks you to upload `nlu_colab.zip`. To skip
   the prompt, drag it into the Files panel (`/content/`) or put it in `MyDrive/CS425/` beforehand.

The model is saved to Google Drive (`MyDrive/CS425/checkpoints/nlu_joint_best/`) and the results to
`MyDrive/CS425/checkpoints/nlu_results/`; the results are also downloaded as `nlu_results.zip` at the end of the Test
section. After changing the code or the data, make a new zip and upload it again.

**Expected time on a T4:** setup and downloads 1–3 min, then the training sweep (6 runs of about 2–3 min each, every
run evaluated on test). **About 20 min in all.** A single run takes 3–5 min.

The last section tries the NLU on your own messages.

In [ ]:
import os, sys, shutil, subprocess

ZIP_NAME = "nlu_colab.zip"                           # made by python nlu/make_colab_zip.py
DRIVE_ZIP = f"/content/drive/MyDrive/CS425/{ZIP_NAME}"
DRIVE_CKPT_DIR = "/content/drive/MyDrive/CS425/checkpoints"
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # a few batches only, to check the notebook runs end to end

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    root = "/content/CS425-Project"
    os.chdir("/content")
    zip_path = next((p for p in (f"/content/{ZIP_NAME}", DRIVE_ZIP) if os.path.exists(p)), None)
    if zip_path is None:
        print(f"upload {ZIP_NAME} (python nlu/make_colab_zip.py makes it)")
        zip_path = os.path.join("/content", next(iter(files.upload())))
    shutil.rmtree(root, ignore_errors=True)           # a fresh copy, so an older upload never mixes in
    shutil.unpack_archive(zip_path, root)
    print("unpacked", zip_path, "->", root)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "sentence-transformers",
                    "pyyaml", "pronouncing", "wordfreq"], check=True)
    os.chdir(root)
    CKPT_DIR = DRIVE_CKPT_DIR
else:
    if os.path.basename(os.getcwd()) == "nlu":
        os.chdir("..")
    CKPT_DIR = "checkpoints"
if SMOKE_TEST:
    CKPT_DIR = os.path.join(CKPT_DIR, "smoke_test")
sys.path[:0] = [os.getcwd(), os.path.join(os.getcwd(), "data", "negatives")]
print("project:", os.getcwd(), "| checkpoints:", CKPT_DIR, "| smoke test:", SMOKE_TEST)

In [ ]:
import json, random, shutil
from collections import Counter
import torch
from utils import load_config, project_path, set_seed
from nlu.joint_model import NLU, build, encode, load, train
from nlu.train_nlu import evaluate, read_jsonl, sweep, sweep_table

cfg = load_config()
hp = dict(cfg["nlu"])                        # hyperparameters from config.yaml
SEED = set_seed(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = os.path.join(CKPT_DIR, "nlu_joint_best")
RESULTS_DIR = project_path("eval/results/nlu")
if SMOKE_TEST:                               # keep the real results
    RESULTS_DIR = project_path(CKPT_DIR) / "nlu_results"

rows = {s: read_jsonl(project_path(cfg["paths"]["nlu_dir"]) / f"{s}.jsonl") for s in ("train", "val", "test")}
traits = json.load(open(project_path(cfg["paths"]["traits"]), encoding="utf-8"))
heldout = {t for t, v in traits.items() if v["heldout"]}
print("device:", device, torch.cuda.get_device_name() if device == "cuda" else "", "| model:", hp["model"])
print({s: len(v) for s, v in rows.items()}, "| held-out-trait test messages:",
      sum(bool(set(r["traits"]) & heldout) for r in rows["test"]))
print("train intents:", dict(Counter(r["intent"] for r in rows["train"])))

In [ ]:
# One training example: whitespace words, their BIO tags and the word pieces the model sees. Each word's tag sits on
# its first piece; the other pieces are left out of the loss (-100).
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(hp["model"])
r = next(r for r in rows["train"] if r["traits"])
ids, labels, _ = encode(tok, r["tokens"], r["tags"], hp["max_len"])
print(r["intent"], r["traits"])
print(" ".join(f"[{t}]" if g != "O" else t for t, g in zip(r["tokens"], r["tags"])))
print([(p, "-" if l == -100 else ["O", "B", "I"][l]) for p, l in zip(tok.convert_ids_to_tokens(ids), labels)])

## Training: learning rate × seed sweep
One model per learning rate in `LRS` and seed in `SEEDS` (default 3e-5 and 5e-5 × 3 seeds), each up to `EPOCHS`
epochs with early stopping (patience 2), and each evaluated on test. A run keeps the epoch with the best val score, the
mean of val intent accuracy and per-word slot F1 (per word because exact-span F1 jumps between epochs with the loose
gold boundaries). The learning rate with the best mean val score wins, and its run with the best val score becomes the
model. **Test results are only reported, never used to choose.** Two runs with the same settings differ by about 0.04
in exact-span F1, so the table gives mean ± standard deviation over the seeds.

Runs are saved to the Colab disk (`/content/nlu_sweep/`); only the chosen one is copied to Drive, replacing
`nlu_joint_best/` there. About 2–3 min per run on a T4, so **15–20 min** for the default 6. For a single run set
`LRS = [hp["lr"]]` and `SEEDS = [SEED]`.

In [ ]:
LRS = [3e-5, 5e-5]
SEEDS = [42, 43, 44]
EPOCHS = 10                    # early stopping usually ends a run sooner
MAX_BATCHES = None
SWEEP_DIR = "/content/nlu_sweep" if IN_COLAB else os.path.join(CKPT_DIR, "nlu_sweep")
if SMOKE_TEST:
    rows = {s: v[:64] for s, v in rows.items()}
    LRS, SEEDS, EPOCHS, MAX_BATCHES = [5e-5], [42, 43], 1, 3

runs, chosen = sweep(rows, traits, hp, LRS, SEEDS, EPOCHS, device, SWEEP_DIR, max_batches=MAX_BATCHES)
if device == "cuda":
    print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

In [ ]:
sweep_table(runs)
print(f"\nchosen: {chosen['name']} (lr with the best mean val score, then its seed with the best val score)")
history = chosen["history"]
print(f"{'epoch':>5} {'train loss':>10} {'val intent':>10} {'slot F1':>8} {'word F1':>8} {'overlap':>8} {'s':>5}")
for h in history:
    print(f"{h['epoch']:5d} {h['train_loss']:10.4f} {h['val_intent_acc']:10.3f} {h['val_slot_f1']:8.3f} "
          f"{h['val_slot_word_f1']:8.3f} {h['val_slot_overlap_f1']:8.3f} {h['seconds']:5.0f}")
shutil.rmtree(CKPT, ignore_errors=True)
shutil.copytree(chosen["ckpt"], CKPT, ignore=shutil.ignore_patterns("results"))
print("copied", chosen["ckpt"], "->", CKPT)

## Test
The chosen run on the test set (529 messages; the 197 about held-out traits were never seen in training). Writes
`summary.json`, `test_errors.jsonl`, `history.json` and `sweep.json` (every run of the sweep) to `eval/results/nlu/`;
the cells below copy them to Drive and download them.

- **Slot F1:** exact = the predicted span covers exactly the gold words; per word = tagged words; overlap = a span
  counts if it overlaps one on the other side.
- **Normalization** maps the *gold* spans to traits; **end to end** compares the traits found in a message with its
  labels. Lenient counts a trait from the same `CLOSE` group as the label (calm for easygoing).

In [ ]:
model, tok = load(CKPT, device)
best_epoch = json.load(open(os.path.join(CKPT, "labels.json")))["epoch"]
summary = evaluate(model, tok, rows["test"], traits, hp, device, RESULTS_DIR, best_epoch)
shutil.copy(os.path.join(CKPT, "history.json"), RESULTS_DIR / "history.json")
json.dump({"lrs": LRS, "seeds": SEEDS, "epochs": EPOCHS, "chosen": chosen["name"],
           "runs": [{k: v for k, v in r.items() if k != "ckpt"} for r in runs]},
          open(RESULTS_DIR / "sweep.json", "w"), indent=2)

In [ ]:
print("per-intent accuracy:")
for i, a in summary["test"]["per_intent_acc"].items():
    print(f"  {i:14s} {a:.3f}")
print("most common intent confusions (gold -> predicted):")
for k, n in list(summary["test"]["intent_confusions"].items())[:10]:
    print(f"  {k:32s} {n}")

In [ ]:
# Copy the results to Drive and download them as nlu_results.zip; unzip it into eval/results/nlu/ in the repo.
if IN_COLAB:
    dest = os.path.join(CKPT_DIR, "nlu_results")
    shutil.copytree(RESULTS_DIR, dest, dirs_exist_ok=True)
    print("copied", sorted(os.listdir(dest)), "to", dest)
    files.download(shutil.make_archive("/content/nlu_results", "zip", RESULTS_DIR))

## Errors
A sample of test messages where the intent or the spans differ from the labels.

In [ ]:
errors = read_jsonl(RESULTS_DIR / "test_errors.jsonl")
print(len(errors), "test messages with an intent or span difference")
for e in random.Random(SEED).sample(errors, min(15, len(errors))):
    intent = e["intent"] if e["intent"] == e["pred_intent"] else f"{e['intent']} -> {e['pred_intent']}"
    print(f"- {e['text'][:110]}\n    {intent} | gold {e['gold_spans']} | pred {e['pred_spans']}")

## Try it
`NLU.parse` is what the dialogue manager calls: intent, confidence, and each span with its trait (`how` = key, synonym,
nearest or unknown; unknown keeps the `nearest` trait).

In [ ]:
nlu = NLU(model, tok, device=device)
for m in ["hey there!", "roast me, i'm super confident and i never stop going to the gym",
          "people say i'm always on time for everything", "another one pls", "make it meaner",
          "ok that was too much, go easy", "roast me", "what's the weather tomorrow", "bye!",
          "i'm the funniest guy in the office and i read like 50 books a year"]:
    r = nlu.parse(m)
    print(f"{m!r:72} -> {r['intent']} ({r['confidence']}) "
          f"{[(t['span'], t['trait'], t['how']) for t in r['traits']]}")

In [ ]:
nlu.parse("roast me, i plan every trip down to the minute and still get lost")